# NutriScope · resumable experiments
No benchmark results are supplied. Use a Colab GPU runtime (12–16 GB), Python 3.12, and licensed datasets. Run one experiment at a time; RF-DETR Base may need a smaller resolution or Colab Pro if it exceeds memory. Never split by crops or adjacent video frames. You must prepare reviewed dataset manifests before training.

In [ ]:
import sys
print(sys.version)
assert sys.version_info[:2] == (3, 12), "Use a Python 3.12 runtime matching the project"
from google.colab import drive
drive.mount("/content/drive")

Upload the complete project folder to `/content/drive/MyDrive/nutriscope`. Copy code and a selected dataset subset to local Colab storage for speed; write checkpoints to Drive. This notebook does not redistribute data or bypass dataset access terms.

In [ ]:
from pathlib import Path
import shutil, os
source = Path("/content/drive/MyDrive/nutriscope")
assert (source / "requirements-ml.txt").exists(), "Upload the project to Drive first"
shutil.copytree(source, "/content/nutriscope", dirs_exist_ok=True, ignore=shutil.ignore_patterns(".venv", "data", "models", "reports"))
shutil.copytree(source / "data", "/content/nutriscope/data", dirs_exist_ok=True, ignore=shutil.ignore_patterns("raw", "processed", "*.sqlite*"))
os.chdir("/content/nutriscope")

In [ ]:
%pip install -r requirements-ml.lock
!nvidia-smi

Follow `data/DATASETS.md`. Prepare `/content/drive/MyDrive/nutriscope-data/raw.jsonl`, `taxonomy.json`, and a measured-weight manifest. Ground-truth item masks are not part of Nutrition5k. Train dish-level weight models unless you collect per-item labels. The same immutable dish IDs must be used across all task manifests.

In [ ]:
import subprocess
data = Path("/content/drive/MyDrive/nutriscope-data")
runs = Path("/content/drive/MyDrive/nutriscope-runs")
runs.mkdir(exist_ok=True)
assert (data / "raw.jsonl").exists() and (data / "taxonomy.json").exists(), "Prepare and review dataset manifests first"
subprocess.run([sys.executable,"-m","ml.data","--manifest",str(data/"raw.jsonl"),"--taxonomy",str(data/"taxonomy.json"),"--output",str(data/"manifest.jsonl")],check=True)

In [ ]:
subprocess.run([sys.executable,"-m","ml.export_coco","--manifest",str(data/"manifest.jsonl"),"--out","/content/coco"],check=True)

Run each detector separately. Batch size is 1 with eight accumulation steps. To resume a detector, add `--resume /content/drive/MyDrive/nutriscope-runs/detector-small/checkpoint.pth` using the actual checkpoint emitted by your RF-DETR version.

In [ ]:
for size in ["small", "base"]:
    subprocess.run([sys.executable,"-m","ml.train_detector","--data","/content/coco","--size",size,"--out",str(runs/f"detector-{size}"),"--epochs","30"],check=True)

In [ ]:
for backbone in ["resnet50","efficientnetv2","convnext-tiny"]:
    subprocess.run([sys.executable,"-m","ml.train_classifier","--manifest",str(data/"manifest.jsonl"),"--backbone",backbone,"--out",str(runs/f"classifier-{backbone}"),"--epochs","20","--resume"],check=True)

Weight training requires a measured `weight-manifest.jsonl` with the same group split. Create this from Nutrition5k dish-level labels and aligned geometry, without pseudo-labeling per-item mass.

In [ ]:
weight_manifest=data/"weight-manifest.jsonl"
assert weight_manifest.exists(), "Prepare measured-weight manifest"
for backbone in ["resnet50","efficientnetv2","convnext-tiny"]:
    subprocess.run([sys.executable,"-m","ml.train_classifier","--manifest",str(weight_manifest),"--backbone",backbone,"--out",str(runs/f"cnn-{backbone}"),"--regression","--resume"],check=True)
    subprocess.run([sys.executable,"-m","ml.predict_cnn","--manifest",str(weight_manifest),"--checkpoint",str(runs/f"cnn-{backbone}/best.pt"),"--output",str(runs/f"cnn-{backbone}.jsonl")],check=True)

In [ ]:
for backbone in ["resnet50","efficientnetv2","convnext-tiny"]:
    subprocess.run([sys.executable,"-m","ml.extract_features","--manifest",str(weight_manifest),"--checkpoint",str(runs/f"classifier-{backbone}/best.pt"),"--output",str(runs/f"features-{backbone}.npz")],check=True)
    subprocess.run([sys.executable,"-m","ml.train_weight","--features",str(runs/f"features-{backbone}.npz"),"--cnn-predictions",str(runs/f"cnn-{backbone}.jsonl"),"--out",str(runs/f"weight-{backbone}")],check=True)

Use `ml.evaluate` on untouched test predictions, including measured calories and carbs, and COCO detector predictions. Record CPU hardware, thread count, warm-up, model artifacts and full-pipeline timings. Populate `reports/model_comparison.csv`; report overall and per-class weight errors and conformal coverage. Do not recommend a winner before validation results exist. The current feature path uses supplied calibrated geometry, not end-to-end predicted masks; disclose this oracle-geometry distinction when reporting results.